模型调用失败时重试，策略和工具调用的重试一样，都是基于指数退避算法。

In [18]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[
        ModelRetryMiddleware(
            max_retries=6,
            backoff_factor=2.0,
            initial_delay=1.0,
            max_delay=10.0,
            on_failure="continue",
            jitter=False, #关闭抖动
        ),
    ],
)

response = agent.invoke({
    "messages": [HumanMessage("你好")],
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？
